# utilities

> Utilities used in measuring the ICL in images of galaxy clusters.

In [ ]:
# | default_exp utilities

In [ ]:
# | export
import numpy as np
from astropy import units as u
from astropy.cosmology import FlatLambdaCDM
from astropy.nddata import CCDData, Cutout2D
from astropy.coordinates import SkyCoord
from astropy.wcs import WCS
from astropy.io.fits import Header
from astropy.stats import mad_std
from regions import PixCoord, RectanglePixelRegion, RectangleSkyRegion
from shapely.geometry import Polygon


from nicl import ezgal

In [ ]:
# | export


def calc_kcorr(z, filter):
    """Calculate the kcorrection from restframe B-band to an observation in `filter`.

    To convert the conventional B-band surface brightness limit to the observed band we
    use [ezgal](https://github.com/cmancone/easyGalaxy). As ezgal does not work correctly
    when installed via pip, and needs some data files and tweaks, we include a copy with nicl.

    """
    model = ezgal.model("bc03_ssp_z_0.02_chab.model")
    model.set_cosmology(Om=0.3, Ol=0.7, h=0.70, w=-1)
    zf = 3  # Redshift of formation, as in Furnell+21
    rest_b = model.get_absolute_mags(zf=zf, filters="B.dat", zs=z)
    obs_i = model.get_observed_absolute_mags(zf=zf, filters=filter, zs=z)
    kcorr = obs_i - rest_b
    return kcorr * u.mag


def calc_sb_threshold(z, filter, b_band_sb_threshold=25 * u.ABmag):
    k_corr = calc_kcorr(z, filter)
    dimming = 2.5 * np.log10((1 + z) ** 4) * u.mag
    return b_band_sb_threshold + k_corr + dimming


def sb_to_adu(sb, pix_scale, zp=27 * u.ABmag):
    sb_to_mag = sb - 2.5 * np.log10(pix_scale.to_value(u.arcsec) ** 2) * u.mag
    counts = 10 ** (-(sb_to_mag - zp).value / 2.5)
    return counts

In [ ]:
# | export


def get_pixel_scale(img, wcs=None):
    """Calculate the average pixel scale of the supplied image."""
    if wcs is None:
        wcs = image.wcs
    return (
        np.mean(
            [s.to_value("arcsec") for s in wcs.celestial.proj_plane_pixel_scales()]
        )
        * u.arcsec
    )


def get_img_centre_pixel(img):
    return (np.array(img.shape) - 1) / 2


def get_img_centre_world(img, wcs=None):
    """Calculate the world coordinates at the centre of the supplied image."""
    centre_pixel = get_img_centre_pixel(img)
    if wcs is None:
        wcs = image.wcs
    return wcs.pixel_to_world(*centre_pixel)


def distance_from_coord(shape, coord):
    """Calculate the distance from pixel `coord` to each pixel in an image with given `shape`."""
    ij = np.indices(shape)[::-1]
    coord = np.asarray(coord)[:, None, None]
    r = np.sqrt(((ij - coord) ** 2).sum(axis=0))
    return r

In [ ]:
# | export


def physical_to_angular(r_physical, z):
    """Convert a physical distance in the plane of the sky to an angle, for a given redshift `z`."""
    cosmo = FlatLambdaCDM(H0=70, Om0=0.3)
    deg_per_Mpc = 1 / cosmo.kpc_proper_per_arcmin(z).to(u.Mpc / u.deg)
    r_angular = r_physical * deg_per_Mpc
    return r_angular.to(u.deg)

In [ ]:
# | export


def pix2arcmin(
    axis: str,  # the plot axis, 'x' or 'y'
    img: CCDData,  # the image previously displayed with imshow
):
    """Return closures for transforming between pixel coordinates and arcmin offsets.

    These are for use when adding secondary axes to a matplotlib plot created with imshow, e.g.
    `ax.secondary_xaxis('bottom', functions=pix2arcmin('x', img))`.

    The pixel coordinates correspond to the use of `plt.imshow` without specifying an extent.
    Offsets are from the centre of the image.
    """
    idx = 0 if axis == "y" else 1
    pixel_size = get_pixel_scale(img)
    centre = get_img_centre_pixel(img)

    def forward(pix):
        return ((pix - centre[axis]) * pixel_size).to_value(u.arcmin)

    def inverse(arcmin):
        return ((arcmin * u.arcmin / pixel_size) + centre[idx]).to_value()

    return forward, inverse


def pix2Mpc(axis, img, z):
    """Return closures for transforming between pixel coordinates and Mpc offsets.

    These are for use when adding secondary axes to a matplotlib plot created with `plt.imshow`, e.g.
    `ax.secondary_xaxis('bottom', functions=pix2Mpc('x', img, z))`.

    The pixel coordinates correspond to the use of `plt.imshow` without specifying an extent.
    Offsets are from the centre of the image.
    """
    p2a, a2p = pix2arcmin(axis, img)
    arcmin_per_Mpc = physical_to_angular(1 * u.Mpc, z).to_value(u.arcmin)

    def forward(pix):
        return p2a(pix) / arcmin_per_Mpc

    def inverse(Mpc):
        return a2p(Mpc * arcmin_per_Mpc)

    return forward, inverse

In [ ]:
# | export


def get_cutout(image, size, position=None, mask=None, wcs=None):
    if position is None:
        position = get_img_centre_pixel(image).astype(int)
    if wcs is None:
        wcs = image.wcs
    cutout = Cutout2D(image, position, size, wcs=wcs, mode="partial").data
    if mask is not None:
        mask = Cutout2D(mask, position, size, wcs=wcs, mode="partial").data
        return cutout, mask
    else:
        return cutout

In [ ]:
# | export


def maybe_to_value(x, unit):
    """Convert `x` to a value in specified `unit`, assuming in correct unit if `x` is not q Quantity."""
    return u.Quantity(x, unit).to_value(unit)

In [ ]:
# | export


def parse_input_for_skycoord(skycoord):
    """Parse input to return a SkyCoord object.

    Parameters
    ----------
    skycoord : str or SkyCoord
        The input sky coordinate which can be either a string or a SkyCoord object.

    Returns
    -------
    SkyCoord
        A SkyCoord object representing the input sky coordinate.

    """
    if isinstance(skycoord, str):
        return SkyCoord(skycoord, unit=(u.hourangle, u.deg))
    elif isinstance(skycoord, SkyCoord):
        return skycoord
    else:
        raise ValueError("Input must be a string or a SkyCoord object.")


def parse_input_for_angular_size(angular_size, duplicate=False):
    """Parse input to return an angular size Quantity.

    Parameters
    ----------
    angular_size : Quantity, list, tuple, ndarray, or str
        The input angular size which can be a Quantity object, a list, tuple,
        ndarray of values, or a string representing the angular size.
    duplicate : bool or int, optional
        If True or a number equivalent to True, the function will return a list
        containing duplicated angular size values. The total number of elements
        will be `duplicate + 1`. True is equal to 1 in this case. Only applicable
        if the input is a scalar. Useful for turning a single angular size into
        a pair or more. Default is False.

    Returns
    -------
    Quantity or list of Quantity
        The parsed angular size as a Quantity object. If `duplicate` is True,
        returns a list of duplicated Quantity objects.

    """
    # note that u.Quantity is also an instance of ndarray
    if isinstance(angular_size, np.ndarray):
        angular_size = np.atleast_1d(angular_size)
        n_elem = angular_size.size
        match n_elem:
            case 1:
                angular_size = u.Quantity(angular_size[0], unit=u.deg)
            case x if x > 1:
                return [u.Quantity(x, unit=u.deg) for x in angular_size.flat]
            case 0:
                raise ValueError("Input must contain at least one element.")
    elif isinstance(angular_size, (list, tuple)):
        n_elem = len(angular_size)
        match n_elem:
            case 1:
                angular_size = u.Quantity(angular_size[0], unit=u.deg)
            case x if x > 1:
                return [u.Quantity(x, unit=u.deg) for x in angular_size]
            case 0:
                raise ValueError("Input must contain at least one element.")
    elif isinstance(angular_size, (int, float, str)):
        # unit=u.deg is safe because it does not make a difference if the input string has a unit
        angular_size = u.Quantity(angular_size, unit=u.deg)
    else:
        raise ValueError(
            "Input must be a string/int/float, a list/tuple/ndarray, or a Quantity object."
        )
    if duplicate:
        return (int(duplicate) + 1) * [angular_size]
    else:
        return angular_size

In [ ]:
# | export


def does_image_overlap_with_skyregion(hdr, sky_reg, threshold=0.0):
    """Check if an image overlaps with a sky region.

    Parameters
    ----------
    hdr : dict-like
        The FITS header of the image to check for overlap with the sky region.
        Must be compatible with WCS creation and refer to a 2D image.
    sky_reg : RectangleSkyRegion
        The sky region to check for overlap with the image.
    threshold : float, optional
        Minimum fraction of overlap required to return True. Default is 0.0,
        which means any overlap will return True.

    Returns
    -------
    bool
        True if the image overlaps with the sky region by more than the threshold,
        False otherwise.
    """
    # Check if header is a dict-like object
    if not isinstance(hdr, (dict, Header)) or not hasattr(hdr, "keys"):
        raise ValueError("Input header must be a dict-like object")

    # Check if the image is 2D
    if "NAXIS" not in hdr:
        raise ValueError("Header doesn't contain NAXIS keyword")
    if hdr["NAXIS"] != 2:
        raise ValueError("This function only works with 2D images (NAXIS=2)")
    if "NAXIS1" not in hdr or "NAXIS2" not in hdr:
        raise ValueError(
            "Header must contain NAXIS1 and NAXIS2 keywords for a 2D image"
        )

    # Try to extract WCS information from header
    try:
        wcs = WCS(hdr)
        if not wcs.has_celestial:
            raise ValueError("Header does not contain valid celestial WCS information.")
    except Exception as e:
        raise ValueError(f"Could not create WCS from header: {e}")

    if not isinstance(sky_reg, RectangleSkyRegion):
        raise ValueError("sky_reg must be a RectangleSkyRegion object.")

    nx = hdr["NAXIS1"]
    ny = hdr["NAXIS2"]
    pix_reg_from_sky = sky_reg.to_pixel(wcs)
    pix_reg_from_img = RectanglePixelRegion(
        center=PixCoord((nx - 1) / 2, (ny - 1) / 2), width=nx, height=ny
    )
    # switch to shapely for intersection calculation because regions has no efficient implementation for that
    sky_polygon = Polygon(pix_reg_from_sky.corners)
    img_polygon = Polygon(pix_reg_from_img.corners)
    if not img_polygon.intersects(sky_polygon):
        return False
    num_pix_overlap = img_polygon.intersection(sky_polygon).area
    num_pix_img = nx * ny
    num_pix_reg = sky_polygon.area
    frac1 = num_pix_overlap / num_pix_img
    frac2 = num_pix_overlap / num_pix_reg
    return frac1 > threshold or frac2 > threshold

In [ ]:
# | export


def bootstrap_median_error(data, errors, nboot=1000, axis=None):
    """
    Estimate the error of the median along a given axis using bootstrapping.

    Parameters:
        data (np.ndarray): Array of measured values.
        errors (np.ndarray): Corresponding 1-sigma uncertainties (same shape as data).
        nboot (int): Number of bootstrap iterations.
        axis (int or None): Axis along which to compute the median.
                            If None, the array is flattened.

    Returns:
        np.ndarray or float: Estimated standard error of the median along the specified axis.
    """
    data = np.asarray(data)
    errors = np.asarray(errors)

    # If axis is not None and the shape along that axis is 1, return the errors directly
    if axis is not None and data.shape[axis] == 1:
        # Need to squeeze out the singular dimension
        return np.squeeze(errors, axis=axis)
    # If axis is None, check if they have more than one element
    elif axis is None and data.size <= 1:
        return np.squeeze(errors)

    # Create all synthetic bootstrap samples at once.
    # The simulated array will have shape (nboot, ...) where ... is data.shape.
    simulated = np.random.normal(loc=data, scale=errors, size=(nboot,) + data.shape)

    if axis is None:
        # Flatten each bootstrap sample and compute the median along the flattened axis.
        medians = np.nanmedian(simulated.reshape(nboot, -1), axis=1)
    else:
        # Adjust the axis by 1 because the bootstrap dimension is at index 0.
        medians = np.nanmedian(simulated, axis=axis + 1)

    # Compute the standard deviation of the medians along the bootstrap dimension.
    return np.nanstd(medians, axis=0)


def fast_median_error(data, axis=None):
    """
    Estimate the error of the median along a given axis using an analytical approach.

    Parameters:
        data (np.ndarray): Array of measured values.
        axis (int or None): Axis along which to compute the median.
                            If None, the array is flattened.

    Returns:
        np.ndarray or float: Estimated standard error of the median along the specified axis.
    """
    data = np.asarray(data)

    # If axis is not None and the shape along that axis is 1, raise an error
    if axis is not None and data.shape[axis] == 1:
        raise ValueError(
            "Cannot compute median error along axis with only one element."
        )
    # If axis is None, check if they have more than one element
    elif axis is None and data.size <= 1:
        raise ValueError(
            "Cannot compute median error for an array with only one element."
        )

    # Compute the standard error of the median using a normal approximation 1.253*sigma/sqrt(n)
    return (
        1.253 * mad_std(data, axis=axis, ignore_nan=True) / np.sqrt(np.isfinite(data).sum(axis=axis))
    )


In [ ]:
# | hide
import nbdev

nbdev.nbdev_export()